``SequenceFeature().dipeptide_composition()`` builds a **baseline feature set** for a prediction model. For each sequence it counts the fraction of each of the 400 ordered adjacent canonical amino-acid pairs over a span, giving the ``(n_seq, 400)`` matrix ``X`` — the sequence's dipeptide composition (DPC). DPC captures local sequential order (which residue follows which) that plain amino-acid composition discards, while still being position-agnostic. Its purpose is **comparison**: fit the same classifier on this ``X`` and on a :class:`CPP` ``feature_matrix``, and compare the scores to see how much CPP's positional Part-Split-Scale features add over a plain dipeptide-frequency encoding. Here we load the ``DOM_GSEC`` example dataset (see [Breimann25]_):

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False
df_seq = aa.load_dataset(name="DOM_GSEC")
sf = aa.SequenceFeature()

By default (``list_parts=None``) the whole TMD-JMD span (``jmd_n`` + ``tmd`` + ``jmd_c``) is used. Only the 20 canonical amino acids are counted; gap symbols (``'-'``) and other non-canonical residues are dropped before pairing, so each row with at least two canonical residues sums to 1:

In [2]:
X = sf.dipeptide_composition(df_seq=df_seq)
print(f"n samples: {X.shape[0]}")
print(f"n dipeptides: {X.shape[1]}")
print(f"Shape of X: {X.shape}")
print(f"Row sums (first 3): {X[:3].sum(axis=1)}")

n samples: 126
n dipeptides: 400
Shape of X: (126, 400)
Row sums (first 3): [1. 1. 1.]


With ``return_df=True`` the matrix is returned as a labeled ``pd.DataFrame`` (rows indexed by protein entry, one column per ordered amino-acid pair ``AA, AC, ..., YY``):

In [3]:
df_dipeptide_composition = sf.dipeptide_composition(df_seq=df_seq, return_df=True)
aa.display_df(df_dipeptide_composition, n_rows=10, show_shape=True)

DataFrame shape: (126, 400)


Use ``list_parts`` to count adjacent pairs only over selected sequence parts. For example, restrict the baseline to the TMD:

In [4]:
X_tmd = sf.dipeptide_composition(df_seq=df_seq, list_parts="tmd")
print(f"Shape of X (TMD only): {X_tmd.shape}")

Shape of X (TMD only): (126, 400)


Multiple parts are concatenated into one span before pairing, so an adjacent pair may cross the boundary between two parts (the last residue of one part pairs with the first residue of the next):

In [5]:
X_jmd = sf.dipeptide_composition(df_seq=df_seq, list_parts=["jmd_n", "jmd_c"])
print(f"Shape of X (JMD_N + JMD_C): {X_jmd.shape}")

Shape of X (JMD_N + JMD_C): (126, 400)
